In [ ]:
## ============================================
# MODEL CREATION
# ============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import efficientnet_b2
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
from torch.cuda.amp import autocast, GradScaler

# -----------------------
# CBAM Block (Attention)
# -----------------------
class CBAMBlock(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super(CBAMBlock, self).__init__()
        # Channel Attention
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        self.sigmoid_channel = nn.Sigmoid()
        # Spatial Attention
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        # Channel Attention
        avg_out = self.fc(self.avg_pool(x).view(b, c))
        max_out = self.fc(self.max_pool(x).view(b, c))
        channel_out = self.sigmoid_channel(avg_out + max_out).view(b, c, 1, 1)
        x = x * channel_out
        # Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = torch.cat([avg_out, max_out], dim=1)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(spatial_out))
        x = x * spatial_out
        return x

# ---------------------------------------------------------
# Basic Conv Block
# ---------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel=3, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel, padding=1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout2d(dropout)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.dropout(x)
        return x


# ---------------------------------------------------------
# Residual Block (2 conv blocks + skip)
# ---------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels)
        self.conv2 = ConvBlock(channels, channels)

    def forward(self, x):
        identity = x
        x = self.conv1(x)
        x = self.conv2(x)
        return x + identity


# ---------------------------------------------------------
# BlockTwoSkip (2 residual blocks + skip)
# ---------------------------------------------------------
class BlockTwoSkip(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.res1 = ResidualBlock(channels)
        self.res2 = ResidualBlock(channels)

    def forward(self, x):
        identity = x
        x = self.res1(x)
        x = self.res2(x)
        return x + identity

class GraphConv(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x, adj):
        # x: [B, N, C]
        # adj: [N, N]
        out = torch.matmul(adj, x)          # message passing
        out = self.linear(out)
        return F.relu(out)

def build_grid_adj(h, w, device):
    n = h * w
    adj = torch.zeros(n, n, device=device)

    for i in range(h):
        for j in range(w):
            idx = i * w + j
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < h and 0 <= nj < w:
                    nidx = ni * w + nj
                    adj[idx, nidx] = 1

    # self-loop + normalization
    adj += torch.eye(n, device=device)
    deg = adj.sum(dim=1, keepdim=True)
    adj = adj / deg
    return adj

# ---------------------------------------------------------
# Full Model with densenet121 Backbone
# ---------------------------------------------------------
class FERModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # -------- DenseNet Backbone --------
        backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1
        )
        self.backbone = backbone.features   # [B, 1024, 7, 7]
        in_channels = 1024


        # Feature blocks (كما هي)
        self.conv1a = ConvBlock(in_channels, 128)
        mid_channels = 128   # 🔥 مناسب جدًا لـ JAFFE

        # -------------------
        # Feature extractor
        # -------------------
        self.conv1 = ConvBlock(in_channels, mid_channels, dropout=0.2)
        self.block1 = BlockTwoSkip(mid_channels)

        self.conv2 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block2 = BlockTwoSkip(mid_channels)

        self.cbam = CBAMBlock(mid_channels)

        self.conv3 = ConvBlock(mid_channels, mid_channels, dropout=0.2)
        self.block3 = BlockTwoSkip(mid_channels)

        self.gap = nn.AdaptiveAvgPool2d(1)
        # GNN
        
        self.gnn1 = GraphConv(mid_channels, mid_channels)
        self.gnn2 = GraphConv(mid_channels, mid_channels)
        
        self.gnn_fc = nn.Linear(mid_channels, mid_channels)
        # -------------------
        # Classifier
        # -------------------
        self.classifier = nn.Sequential(
            nn.Linear(mid_channels, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)
    
        x = self.conv1(x)
        x = self.block1(x)
    
        x = self.conv2(x)
        x = self.block2(x)
    
        x = self.cbam(x)
        x = self.conv3(x)
        x = self.block3(x)
    
        # --------------------
        # CNN → Graph
        # --------------------
        B, C, H, W = x.shape
        nodes = x.view(B, C, H*W).permute(0, 2, 1)  # [B, N, C]
    
        adj = build_grid_adj(H, W, x.device)
    
        nodes = self.gnn1(nodes, adj)
        nodes = self.gnn2(nodes, adj)
    
        # Graph pooling
        gnn_feat = nodes.mean(dim=1)   # [B, C]
        gnn_feat = self.gnn_fc(gnn_feat)
    
        # --------------------
        # Final classifier
        # --------------------
        logits = self.classifier(gnn_feat)
        return logits


In [ ]:
def freeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = False

def unfreeze_backbone(model):
    for p in model.backbone.parameters():
        p.requires_grad = True


In [ ]:
class EarlyStopping:
    def __init__(self, patience=15):
        self.patience = patience
        self.counter = 0
        self.best_acc = 0

    def step(self, val_acc):
        if val_acc > self.best_acc:
            self.best_acc = val_acc
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


In [ ]:
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

# -------------------
# Training Function
# -------------------
def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=50):
    model.to(device)
    best_val_acc = 0.0
    scaler = GradScaler()
    #early_stopping = EarlyStopping(patience=PATIENCE)

    best_val_acc = 0.0
    freeze_backbone(model)
    for epoch in range(epochs):
        if epoch == FREEZE_EPOCHS:
            unfreeze_backbone(model)
            print("🔓 Backbone Unfrozen")
        
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            train_correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

        train_loss /= total
        train_acc = 100. * train_correct / total

        # ---- Validation ----
        model.eval()
        val_loss, val_correct, total_val = 0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]"):
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(labels).sum().item()
                total_val += labels.size(0)

        val_loss /= total_val
        val_acc = 100. * val_correct / total_val

        print(f"\nEpoch [{epoch+1}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print(f"✅ Best model saved with Val Acc: {val_acc:.2f}%")

    print("Training Finished! Best Val Acc: {:.2f}%".format(best_val_acc))
    return model


In [ ]:
def set_seed(seed=42):
    import random, numpy as np, torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
from torchvision import transforms

# ── CK+ Dataset Class ──────────────────────────────────────────────────────
class CKPlusDataset(Dataset):
    """
    هيكل مجلدات CK+:
    CK+/
    ├── angry/
    │   ├── S005_001_00000011.png
    │   └── ...
    ├── contempt/
    ├── disgust/
    ├── fear/
    ├── happy/
    ├── sadness/
    └── surprise/
    """

    EMOTIONS = ['AN', 'DI', 'FE',
                'HA', 'NE', 'SA', 'SU']

    def __init__(self, root_dir, transform=None):
        self.root_dir  = root_dir
        self.transform = transform
        self.samples   = []   # list of (img_path, label)
        self.subjects  = []   # subject ID per sample (for LOSO option)

        for label, emotion in enumerate(self.EMOTIONS):
            emotion_dir = os.path.join(root_dir, emotion)
            if not os.path.isdir(emotion_dir):
                continue
            for fname in sorted(os.listdir(emotion_dir)):
                if fname.lower().endswith(('.png','.jpg','.jpeg')):
                    fpath = os.path.join(emotion_dir, fname)
                    self.samples.append((fpath, label))
                    # CK+ filenames: S005_001_... → subject = S005
                    subject = fname.split('_')[0]
                    self.subjects.append(subject)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# ── Transforms ──────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import numpy as np

def train_one_fold(model, train_loader, test_loader,
                   fold_num, device,
                   num_epochs=50, patience=15):
    """
    يدرّب ويقيّم fold واحد.
    Returns: best_acc, best_f1, all_preds, all_labels
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_acc     = 0.0
    best_f1      = 0.0
    best_preds   = []
    best_labels  = []
    no_improve   = 0

    print(f"\n{'='*50}")
    print(f"  Fold {fold_num} — Training")
    print(f"{'='*50}")

    for epoch in range(1, num_epochs + 1):

        # ── Train ────────────────────────────────────────
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item() * imgs.size(0)
            preds       = outputs.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)

        scheduler.step()
        train_acc  = correct / total
        train_loss = train_loss / total

        # ── Evaluate ─────────────────────────────────────
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs   = imgs.to(device)
                outputs = model(imgs)
                preds   = outputs.argmax(dim=1).cpu().tolist()
                all_preds  .extend(preds)
                all_labels .extend(labels.tolist())

        acc = accuracy_score(all_labels, all_preds) * 100
        f1  = f1_score(all_labels, all_preds,
                       average='weighted') * 100

        print(f"Epoch {epoch:03d}/{num_epochs} | "
              f"Loss={train_loss:.4f} | "
              f"TrainAcc={train_acc*100:.2f}% | "
              f"TestAcc={acc:.2f}% | F1={f1:.2f}%")

        # ── Early Stopping ────────────────────────────────
        if acc > best_acc:
            best_acc    = acc
            best_f1     = f1
            best_preds  = all_preds.copy()
            best_labels = all_labels.copy()
            no_improve  = 0
            # احفظ أحسن weights
            torch.save(model.state_dict(),
                       f'best_fold{fold_num}.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  ⏹ Early stopping at epoch {epoch}")
                break

    print(f"\n  ✅ Fold {fold_num} Best Acc = {best_acc:.2f}% | "
          f"F1 = {best_f1:.2f}%")

    return best_acc, best_f1, best_preds, best_labels

In [ ]:
from sklearn.model_selection import StratifiedKFold
import copy

def run_5fold_ckplus(root_dir, device, num_classes=7):
    """
    يشغّل 5-Fold Cross-Validation كاملة على CK+
    """

    # ── Load full dataset ────────────────────────────────
    full_dataset = CKPlusDataset(root_dir, transform=train_transform)

    # الـ labels لكل sample (مطلوبة لـ StratifiedKFold)
    all_labels = [full_dataset.samples[i][1]
                  for i in range(len(full_dataset))]

    print(f"📦 Total samples: {len(full_dataset)}")
    print(f"📊 Classes: {CKPlusDataset.EMOTIONS}")

    # ── Stratified 5-Fold ────────────────────────────────
    # Stratified = كل fold فيها نفس نسبة كل class
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_results = []   # (acc, f1) لكل fold
    all_preds_combined  = []
    all_labels_combined = []

    for fold, (train_idx, test_idx) in enumerate(
            skf.split(range(len(full_dataset)), all_labels), start=1):

        print(f"\n{'━'*60}")
        print(f"  FOLD {fold}/5 | "
              f"Train={len(train_idx)} | Test={len(test_idx)}")
        print(f"{'━'*60}")

        # ── Samplers ─────────────────────────────────────
        # train dataset: بيستخدم train_transform
        # test dataset:  بيستخدم test_transform (بدون augmentation)
        train_dataset = CKPlusDataset(root_dir,
                                       transform=train_transform)
        test_dataset  = CKPlusDataset(root_dir,
                                       transform=test_transform)

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            sampler=SubsetRandomSampler(train_idx),
            num_workers=2,
            pin_memory=True,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            sampler=SubsetRandomSampler(test_idx),
            num_workers=2,
            pin_memory=True,
        )

        # ── Fresh model لكل fold ──────────────────────────
        model = FERModel(num_classes=num_classes).to(device)

        # ── Train ─────────────────────────────────────────
        acc, f1, preds, labels = train_one_fold(
            model, train_loader, test_loader,
            fold_num=fold,
            device=device,
            num_epochs=50,
            patience=50,
        )

        fold_results.append((acc, f1))
        all_preds_combined .extend(preds)
        all_labels_combined.extend(labels)

    # ────────────────────────────────────────────────────
    # النتائج النهائية
    # ────────────────────────────────────────────────────
    accs = [r[0] for r in fold_results]
    f1s  = [r[1] for r in fold_results]

    print(f"\n{'═'*60}")
    print(f"  5-FOLD CROSS-VALIDATION RESULTS — CK+")
    print(f"{'═'*60}")
    for i, (acc, f1) in enumerate(fold_results, 1):
        print(f"  Fold {i}: Acc={acc:.2f}%  F1={f1:.2f}%")
    print(f"{'─'*60}")
    print(f"  Mean Acc : {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")
    print(f"  Mean F1  : {np.mean(f1s):.2f}% ± {np.std(f1s):.2f}%")
    print(f"  Best Fold: {np.argmax(accs)+1} ({max(accs):.2f}%)")
    print(f"{'═'*60}")

    # Confusion matrix على كل الـ predictions
    cm = confusion_matrix(all_labels_combined, all_preds_combined)
    print(f"\n  Confusion Matrix (all folds combined):")
    print(cm)

    return {
        'fold_results'   : fold_results,
        'mean_acc'       : np.mean(accs),
        'std_acc'        : np.std(accs),
        'mean_f1'        : np.mean(f1s),
        'std_acc'        : np.std(accs),
        'confusion_matrix': cm,
    }

In [ ]:
# ============================================================
# ROC CURVE & AUC (Multi-Class, One-vs-Rest)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
from itertools import cycle


def get_probs_and_labels(model, data_loader, device):
    """Runs the model over a DataLoader and collects softmax probabilities
    plus true labels (needed for ROC/AUC, unlike argmax-only evaluation)."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_probs, axis=0), np.concatenate(all_labels, axis=0)


def plot_multiclass_roc_auc(y_true, y_probs, class_names, title="ROC Curves (One-vs-Rest)"):
    """
    y_true:  1D array of integer class labels, shape [N]
    y_probs: 2D array of predicted probabilities, shape [N, num_classes]
    """
    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    # Handle the edge case of a binary problem (label_binarize returns 1 column)
    if n_classes == 2:
        y_true_bin = np.hstack([1 - y_true_bin, y_true_bin])

    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # ---- Micro-average (aggregate all classes together) ----
    fpr["micro"], tpr["micro"], _ = roc_curve(y_true_bin.ravel(), y_probs.ravel())
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

    # ---- Macro-average (unweighted mean of per-class curves) ----
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"], tpr["macro"] = all_fpr, mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

    # ---- Plot ----
    plt.figure(figsize=(9, 7))
    colors = cycle(["#e6194B", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
                     "#911eb4", "#42d4f4", "#f032e6", "#bfef45", "#fabed4"])
    for i, color in zip(range(n_classes), colors):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                  label=f"{class_names[i]} (AUC = {roc_auc[i]:.3f})")

    plt.plot(fpr["micro"], tpr["micro"], linestyle=":", color="deeppink", lw=3,
              label=f"micro-average (AUC = {roc_auc['micro']:.3f})")
    plt.plot(fpr["macro"], tpr["macro"], linestyle=":", color="navy", lw=3,
              label=f"macro-average (AUC = {roc_auc['macro']:.3f})")

    plt.plot([0, 1], [0, 1], "k--", lw=1, label="Chance (AUC = 0.500)")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right", fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\nPer-class AUC:")
    for i in range(n_classes):
        print(f"  {class_names[i]:>10s}: {roc_auc[i]:.4f}")
    print(f"\n  Macro-average AUC: {roc_auc['macro']:.4f}")
    print(f"  Micro-average AUC: {roc_auc['micro']:.4f}")

    return roc_auc


In [ ]:
# ============================================================
# RUN: ROC / AUC on the held-out test set 
# ============================================================
y_probs, y_true = get_probs_and_labels(best_model, val_loader, device)
roc_auc_scores = plot_multiclass_roc_auc(y_true, y_probs, emotion_labels,
                                          title="DRAG-Net ROC Curves (Test Set)")


In [ ]:
# ============================================================
# Compute full per-fold metrics: Accuracy, Macro Precision,
# Macro Recall, Macro F1-Score  (extends run_5fold_ckplus)
# ============================================================
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import numpy as np
import pandas as pd
root_dir = r"/kaggle/input/datasets/mohamedaliabdeldaym/jaffeall/New Data"

def compute_fold_metrics(preds, labels):
    """Given predictions/labels for one fold, return the 4 metrics (%)."""
    acc = accuracy_score(labels, preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "Accuracy (%)": acc,
        "Macro Precision (%)": precision * 100,
        "Macro Recall (%)": recall * 100,
        "Macro F1-Score (%)": f1 * 100,
    }


def run_5fold_ckplus_full(root_dir, device, num_classes=7):
    """
    Same as run_5fold_ckplus, but stores per-fold predictions/labels
    so Accuracy, Macro Precision, Macro Recall and Macro F1 can all
    be computed for every fold (needed for the summary table).
    """
    full_dataset = CKPlusDataset(root_dir, transform=train_transform)
    all_labels = [full_dataset.samples[i][1] for i in range(len(full_dataset))]

    print(f"📦 Total samples: {len(full_dataset)}")
    print(f"📊 Classes: {CKPlusDataset.EMOTIONS}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    per_fold_metrics = []          # list of dicts (one per fold)
    all_preds_combined = []
    all_labels_combined = []

    for fold, (train_idx, test_idx) in enumerate(
            skf.split(range(len(full_dataset)), all_labels), start=1):

        print(f"\n{'━'*60}")
        print(f"  FOLD {fold}/5 | Train={len(train_idx)} | Test={len(test_idx)}")
        print(f"{'━'*60}")

        train_dataset = CKPlusDataset(root_dir, transform=train_transform)
        test_dataset = CKPlusDataset(root_dir, transform=test_transform)

        train_loader = DataLoader(
            train_dataset, batch_size=16,
            sampler=SubsetRandomSampler(train_idx),
            num_workers=2, pin_memory=True,
        )
        test_loader = DataLoader(
            test_dataset, batch_size=16,
            sampler=SubsetRandomSampler(test_idx),
            num_workers=2, pin_memory=True,
        )

        model = FERModel(num_classes=num_classes).to(device)

        acc, f1, preds, labels = train_one_fold(
            model, train_loader, test_loader,
            fold_num=fold, device=device,
            num_epochs=100, patience=50,
        )

        fold_metrics = compute_fold_metrics(preds, labels)
        per_fold_metrics.append(fold_metrics)

        all_preds_combined.extend(preds)
        all_labels_combined.extend(labels)

        print(f"  Fold {fold} -> "
              f"Acc={fold_metrics['Accuracy (%)']:.2f}% | "
              f"P={fold_metrics['Macro Precision (%)']:.2f}% | "
              f"R={fold_metrics['Macro Recall (%)']:.2f}% | "
              f"F1={fold_metrics['Macro F1-Score (%)']:.2f}%")

    cm = confusion_matrix(all_labels_combined, all_preds_combined)

    return {
        "per_fold_metrics": per_fold_metrics,   # <- feeds the results table
        "confusion_matrix": cm,
    }


In [ ]:
# ============================================================
# Build & display the results table
# (Fold 1-5, Mean ± SD, Best Fold — one row per metric column)
# ============================================================
import pandas as pd


def build_results_table(per_fold_metrics):
    """
    per_fold_metrics: list of dicts, one per fold, each with keys
        'Accuracy (%)', 'Macro Precision (%)', 'Macro Recall (%)', 'Macro F1-Score (%)'
    Returns a pandas DataFrame formatted like:

        Fold | Accuracy (%) | Macro Precision (%) | Macro Recall (%) | Macro F1-Score (%)
        Fold 1 | ...
        ...
        Mean ± SD | ...
        Best Fold | ...
    """
    columns = ["Accuracy (%)", "Macro Precision (%)", "Macro Recall (%)", "Macro F1-Score (%)"]
    df = pd.DataFrame(per_fold_metrics, columns=columns)
    df.index = [f"Fold {i+1}" for i in range(len(df))]

    # ---- Mean ± SD row ----
    mean_sd_row = {
        col: f"{df[col].mean():.2f} ± {df[col].std():.2f}%" for col in columns
    }

    # ---- Best fold row (per column, independently) ----
    best_fold_row = {
        col: f"Fold {df[col].values.argmax() + 1} ({df[col].max():.2f}%)" for col in columns
    }

    # Build a display-friendly version (fold rows as "xx.xx%")
    display_df = df.copy()
    for col in columns:
        display_df[col] = display_df[col].map(lambda v: f"{v:.2f}%")

    display_df.loc["Mean ± SD"] = mean_sd_row
    display_df.loc["Best Fold"] = best_fold_row
    display_df.index.name = "Fold"

    return display_df


# ------------------------------------------------------------------
# Example usage:
results = run_5fold_ckplus_full(root_dir, device, num_classes=7)
results_table = build_results_table(results["per_fold_metrics"])
results_table
#
# If you already have the 5-fold results printed/logged from a previous
# run (as in the summary table), you can build the same table directly
# from those numbers without re-training, e.g.:


#results_table = build_results_table(per_fold_metrics_example)
#results_table


In [ ]:
# ============================================================
# TABLE 11: Aggregated Out-of-Fold (OOF) Performance Metrics
# + Confusion Matrix, across all 5 folds of run_5fold_ckplus_full()
# ============================================================
import numpy as np
import pandas as pd


def compute_oof_aggregated_metrics(cm, class_names):
    """
    Compute Macro Precision / Macro Recall / Macro F1 (%) directly
    from an aggregated (all-folds-combined) confusion matrix.

    cm: [C, C] numpy array, cm[i, j] = actual class i predicted as class j
    """
    cm = np.asarray(cm)
    num_classes = cm.shape[0]

    precisions, recalls, f1s = [], [], []
    for c in range(num_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1        = (2 * precision * recall / (precision + recall)
                     if (precision + recall) > 0 else 0.0)

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    macro_precision = np.mean(precisions) * 100
    macro_recall    = np.mean(recalls) * 100
    macro_f1        = np.mean(f1s) * 100

    metrics_table = pd.DataFrame(
        [{
            "Model / Evaluation Scheme": "Proposed DRAG-Net (Aggregated 5-Fold OOF)",
            "Macro Precision (%)": f"{macro_precision:.2f}%",
            "Macro Recall (%)": f"{macro_recall:.2f}%",
            "Macro F1-Score (%)": f"{macro_f1:.2f}%",
        }]
    ).set_index("Model / Evaluation Scheme")

    cm_table = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_table.index.name = "Actual \\ Predicted"

    return metrics_table, cm_table


# ------------------------------------------------------------------
# Usage:
# results = run_5fold_ckplus_full(root_dir, device, num_classes=7)
metrics_table, cm_table = compute_oof_aggregated_metrics(
results["confusion_matrix"], CKPlusDataset.EMOTIONS)
metrics_table
cm_table
print("Table 11a — Aggregated OOF Performance Metrics")
display(metrics_table)

print("\nTable 11b — Aggregated Confusion Matrix (all 5 folds)")
display(cm_table)
#
# Example using the confusion matrix from your Table 11 image, so you
# can preview the table format immediately without re-training:
#class_names_display = ["Anger", "Contempt", "Disgust", "Fear", "Happy", "Sadness", "Surprise"]



#metrics_table, cm_table = compute_oof_aggregated_metrics(cm_example, class_names_display)

#print("Table 11a — Aggregated OOF Performance Metrics")
#display(metrics_table)

#print("\nTable 11b — Aggregated Confusion Matrix (all 5 folds)")
#display(cm_table)


In [ ]:
# ============================================================
# TABLE 18: Multi-Seed Evaluation (Mean ± 95% CI) and
# Statistical Significance (paired t-test) vs. Proposed DRAG-Net
# ============================================================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import accuracy_score
from scipy import stats
import numpy as np
import pandas as pd


# ---- Baseline model: DenseNet121 backbone only (GAP + linear head) ----
class DenseNet121Baseline(nn.Module):
    """Plain DenseNet121 feature extractor + GAP + classifier
    (no CBAM, no residual blocks, no GNN) -- used as the
    'Backbone-only' ablation baseline in Table 18."""

    def __init__(self, num_classes=7):
        super().__init__()
        backbone = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        self.backbone = backbone.features
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = F.relu(x, inplace=True)
        x = self.gap(x).flatten(1)
        return self.classifier(x)


def train_single_seed(model, train_loader, val_loader, test_loader,
                       device, num_epochs=60, patience=15):
    """Trains one model on one 80/10/10 split and returns test accuracy (%)."""
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_val_acc = 0.0
    best_state = None
    no_improve = 0

    for epoch in range(1, num_epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # ---- validation (for early stopping / best checkpoint) ----
        model.eval()
        v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                preds = model(imgs).argmax(dim=1).cpu().tolist()
                v_preds.extend(preds)
                v_labels.extend(labels.tolist())
        val_acc = accuracy_score(v_labels, v_preds) * 100

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    # ---- final test evaluation using best checkpoint ----
    model.load_state_dict(best_state)
    model.eval()
    t_preds, t_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu().tolist()
            t_preds.extend(preds)
            t_labels.extend(labels.tolist())

    return accuracy_score(t_labels, t_preds) * 100


def run_multi_seed_evaluation(model_class, root_dir, device,
                               seeds=(42, 100, 2024, 777, 999),
                               num_classes=7, num_epochs=100, patience=50):
    """Trains `model_class` from scratch on 5 different random
    80/10/10 splits (one per seed) and returns the list of test
    accuracies (%), one per seed."""
    full_dataset = CKPlusDataset(root_dir, transform=train_transform)
    n = len(full_dataset)
    train_size = int(0.80 * n)
    val_size = int(0.20 * n)
    test_size = n - train_size - val_size

    accs = []
    for seed in seeds:
        print(f"\n{'='*50}\n  Seed {seed} — {model_class.__name__}\n{'='*50}")

        train_ds, val_ds, test_ds = random_split(
            full_dataset, [train_size, val_size, test_size],
            generator=torch.Generator().manual_seed(seed)
        )
        # apply non-augmented transform for val/test
        val_ds.dataset = CKPlusDataset(root_dir, transform=test_transform)
        test_ds.dataset = CKPlusDataset(root_dir, transform=test_transform)

        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
        val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)
        test_loader  = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2)

        model = model_class(num_classes=num_classes).to(device)
        acc = train_single_seed(model, train_loader, val_loader, test_loader,
                                 device, num_epochs=num_epochs, patience=patience)
        print(f"  Seed {seed} -> Test Accuracy = {acc:.2f}%")
        accs.append(acc)

    return accs


def mean_ci95(values):
    """Mean and 95% CI half-width using a t-distribution (small-sample)."""
    values = np.asarray(values, dtype=float)
    mean = values.mean()
    sem = stats.sem(values)
    ci_halfwidth = sem * stats.t.ppf(0.975, df=len(values) - 1)
    return mean, ci_halfwidth


def build_table18(baseline_accs, proposed_accs,
                   baseline_name="DenseNet121 (Backbone-only)",
                   proposed_name="DRAG-Net (Proposed)"):
    """Builds Table 18: Mean ± 95% CI accuracy for baseline & proposed,
    plus a paired t-test p-value (proposed vs. baseline, matched by seed)."""
    base_mean, base_ci = mean_ci95(baseline_accs)
    prop_mean, prop_ci = mean_ci95(proposed_accs)

    t_stat, p_value = stats.ttest_rel(proposed_accs, baseline_accs)

    rows = [
        {"Model / Method": baseline_name,
         "CK+48 Accuracy (Mean \u00b1 95% CI)": f"{base_mean:.2f}% \u00b1 {base_ci:.2f}%",
         "p-value (vs. Proposed)": "\u2014"},
        {"Model / Method": proposed_name,
         "CK+48 Accuracy (Mean \u00b1 95% CI)": f"{prop_mean:.2f}% \u00b1 {prop_ci:.2f}%",
         "p-value (vs. Proposed)": (f"< 0.01 (p = {p_value:.4f})" if p_value < 0.01
                                     else f"p = {p_value:.4f}")},
    ]
    return pd.DataFrame(rows).set_index("Model / Method")


# ------------------------------------------------------------------
# Usage (actually trains 2 models x 5 seeds = 10 full training runs):
#
baseline_accs = run_multi_seed_evaluation(DenseNet121Baseline, root_dir, device)
proposed_accs = run_multi_seed_evaluation(FERModel, root_dir, device)
table18 = build_table18(baseline_accs, proposed_accs)
table18
#
# Example reproducing your Table 18 numbers directly (no re-training),
# so you can preview the exact table format now:
# baseline_accs_example = [67.10, 69.02, 66.85, 70.44, 67.94]   # mean ~68.27% +/- 1.62%
# proposed_accs_example = [81.20, 82.55, 80.90, 82.77, 82.98]   # mean ~81.88% +/- 1.09%

# table18 = build_table18(baseline_accs_example, proposed_accs_example)
# table18


## Table 15b / Table 15a (JAFFE) / Table 17 — ablation from one consistent run
Run cells **0, 5, 9, 10** first (model blocks, `set_seed`, dataset class, transforms), then run the cell below.
All rows of Table 15b, the JAFFE column of Table 15a, and Table 17 are produced with the same protocol as Table 11.

In [ ]:
# ============================================================
# TABLE 15b (+ JAFFE column of Table 15a, + Table 17)
# Ablation of DRAG-Net on JAFFE — ALL rows produced by THIS cell,
# with the SAME protocol used for Table 11:
#   5-fold StratifiedKFold(shuffle=True, random_state=seed)  (sample-wise)
#   batch 16, 100 epochs, AdamW(lr=3e-4, wd=1e-4), CosineAnnealingLR(eta_min=1e-6),
#   CrossEntropy(label_smoothing=0.1), grad-clip 1.0, same train/test transforms.
#
# Requires cells: 0 (model blocks), 5 (set_seed), 9 (CKPlusDataset), 10 (transforms)
# ============================================================
import copy, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from scipy import stats

# ------------------------- settings -------------------------
JAFFE_ROOT   = "D/jaffeall/New Data"
NUM_EPOCHS   = 100
BATCH_SIZE   = 16
MID_CHANNELS = 64            # same value as FERModel in cell 0 for JAFFE
NUM_CLASSES  = 7
TABLE_SEED   = 42            # seed used for Table 15b / 15a (same as Table 11)
T17_SEEDS    = (42, 2024, 999)
RUN_TABLE17  = True          # also produce Table 17 (stages 'a' and 'e' over 3 seeds)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STAGES = {
    "a": "(a) Standalone DenseNet121",
    "b": "(b) DenseNet121 + Projection Layer",
    "c": "(c) + Residual Refinement Blocks",
    "d": "(d) + CBAM",
    "e": "(e) Full DRAG-Net (+ Grid GNN)",
}

# ------------------------- model ----------------------------
class AblationModel(nn.Module):
    """
    a : DenseNet121 features -> ReLU -> GAP -> Linear            (backbone only)
    b : + projection ConvBlock(1024 -> MID) -> GAP -> head
    c : + residual refinement (conv1, block1, conv2, block2, conv3, block3)
    d : + CBAM (between block2 and conv3, exactly as in FERModel)
    e : + grid GraphConv x2 + mean readout + gnn_fc  == FERModel (cell 0)
    The classification head of b-e is identical to FERModel.classifier.
    """
    def __init__(self, stage, num_classes=NUM_CLASSES, mid=MID_CHANNELS):
        super().__init__()
        assert stage in STAGES
        self.stage = stage
        self.backbone = models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
        self.gap = nn.AdaptiveAvgPool2d(1)

        if stage == "a":
            self.head = nn.Linear(1024, num_classes)
            return

        self.conv1 = ConvBlock(1024, mid, dropout=0.2)
        if stage in ("c", "d", "e"):
            self.block1 = BlockTwoSkip(mid)
            self.conv2  = ConvBlock(mid, mid, dropout=0.2)
            self.block2 = BlockTwoSkip(mid)
            self.conv3  = ConvBlock(mid, mid, dropout=0.2)
            self.block3 = BlockTwoSkip(mid)
        if stage in ("d", "e"):
            self.cbam = CBAMBlock(mid)
        if stage == "e":
            self.gnn1   = GraphConv(mid, mid)
            self.gnn2   = GraphConv(mid, mid)
            self.gnn_fc = nn.Linear(mid, mid)

        self.head = nn.Sequential(
            nn.Linear(mid, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.backbone(x)
        if self.stage == "a":
            return self.head(self.gap(F.relu(x)).flatten(1))

        x = self.conv1(x)
        if self.stage in ("c", "d", "e"):
            x = self.block1(x)
            x = self.conv2(x)
            x = self.block2(x)
            if self.stage in ("d", "e"):
                x = self.cbam(x)
            x = self.conv3(x)
            x = self.block3(x)

        if self.stage == "e":
            B, C, H, W = x.shape
            nodes = x.view(B, C, H * W).permute(0, 2, 1)       # [B, N, C]
            adj = build_grid_adj(H, W, x.device)
            nodes = self.gnn2(self.gnn1(nodes, adj), adj)
            feat = self.gnn_fc(nodes.mean(dim=1))
        else:
            feat = self.gap(x).flatten(1)
        return self.head(feat)


# ------------------------- training -------------------------
def predict(model, loader):
    model.eval()
    p, l = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            p.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            l.extend(labels.tolist())
    return p, l


def train_eval_fold(model, train_loader, test_loader, num_epochs=NUM_EPOCHS):
    """Same optimisation as train_one_fold (cell 11). Returns predictions of
    the BEST test epoch (as used for Table 11) and of the LAST epoch."""
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

    best_acc, best, last = -1.0, None, None
    for epoch in range(1, num_epochs + 1):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        preds, labels = predict(model, test_loader)
        acc = accuracy_score(labels, preds) * 100
        if acc > best_acc:
            best_acc, best = acc, (preds, labels)
        last = (preds, labels)
    return best, last


def fold_metrics(preds, labels):
    acc = accuracy_score(labels, preds) * 100
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"Accuracy (%)": acc, "Precision (%)": p * 100,
            "Recall (%)": r * 100, "F1-Score (%)": f1 * 100}


def run_stage(stage, seed):
    """5-fold CV of one ablation stage. Returns per-fold metrics for the
    best and last epoch, plus pooled (OOF) predictions."""
    set_seed(seed)
    ds = CKPlusDataset(JAFFE_ROOT, transform=train_transform)
    y = [lab for _, lab in ds.samples]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    out = {"best": [], "last": [], "oof_best": ([], []), "oof_last": ([], [])}
    for fold, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        t0 = time.time()
        train_loader = DataLoader(CKPlusDataset(JAFFE_ROOT, transform=train_transform),
                                  batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(tr),
                                  num_workers=2, pin_memory=True)
        test_loader = DataLoader(CKPlusDataset(JAFFE_ROOT, transform=test_transform),
                                 batch_size=BATCH_SIZE, sampler=SubsetRandomSampler(te),
                                 num_workers=2, pin_memory=True)
        model = AblationModel(stage).to(device)
        best, last = train_eval_fold(model, train_loader, test_loader)

        for key, (p, l) in (("best", best), ("last", last)):
            out[key].append(fold_metrics(p, l))
            out["oof_" + key][0].extend(p)
            out["oof_" + key][1].extend(l)
        print(f"  stage {stage} | seed {seed} | fold {fold}: "
              f"best Acc={out['best'][-1]['Accuracy (%)']:.2f}% | "
              f"last Acc={out['last'][-1]['Accuracy (%)']:.2f}% | {time.time()-t0:.0f}s")
        del model
        torch.cuda.empty_cache()
    return out


def mean_sd(values):
    v = np.asarray(values, dtype=float)
    return v.mean(), v.std(ddof=1)          # sample SD (same as Table 11)


# ------------------------- Table 15b ------------------------
def build_table15b(results, which="best"):
    rows = []
    for stage, name in STAGES.items():
        df = pd.DataFrame(results[stage][which])
        rows.append({"Ablation Stage/Model Variant": name,
                     **{c: f"{df[c].mean():.2f}" for c in df.columns}})
    return pd.DataFrame(rows).set_index("Ablation Stage/Model Variant")


def build_table15b_with_sd(results, which="best"):
    rows = []
    for stage, name in STAGES.items():
        df = pd.DataFrame(results[stage][which])
        rows.append({"Ablation Stage/Model Variant": name,
                     **{c: "{:.2f} ± {:.2f}".format(*mean_sd(df[c])) for c in df.columns}})
    return pd.DataFrame(rows).set_index("Ablation Stage/Model Variant")


print("=" * 70)
print(f"  TABLE 15b — JAFFE ablation (seed {TABLE_SEED}, 5-fold sample-wise)")
print("=" * 70)
results_15b = {}
for stage in STAGES:
    print(f"\n▶ Running {STAGES[stage]}")
    results_15b[stage] = run_stage(stage, TABLE_SEED)

table15b_best = build_table15b(results_15b, "best")
table15b_last = build_table15b(results_15b, "last")

print("\nTable 15b — mean over 5 folds (BEST test epoch, same rule as Table 11)")
display(table15b_best)
print("\nTable 15b — mean over 5 folds (LAST epoch)")
display(table15b_last)
print("\nTable 15b — mean ± SD over folds (BEST epoch)")
display(build_table15b_with_sd(results_15b, "best"))

# JAFFE column of Table 15a comes from the SAME runs (c, d, e)
table15a_jaffe = pd.DataFrame({
    "Configuration": ["DenseNet121 + CNN FE x3 + Residual Block x6",
                      "+ CBAM Salience Enhancement (SAL)",
                      "+ Grid GNN Processing (Complete DRAG-Net)"],
    "JAFFE Accuracy (%) [best]": [table15b_best.loc[STAGES[s], "Accuracy (%)"] for s in "cde"],
    "JAFFE Accuracy (%) [last]": [table15b_last.loc[STAGES[s], "Accuracy (%)"] for s in "cde"],
}).set_index("Configuration")
print("\nTable 15a — JAFFE column (same runs as Table 15b)")
display(table15a_jaffe)

# Full-model row must match Table 11 / Table 13 / Table 14 (same protocol)
full_df = pd.DataFrame(results_15b["e"]["best"])
print("\nFull DRAG-Net per fold (compare with Table 11):")
display(full_df.round(2))
cm_full = confusion_matrix(*results_15b["e"]["oof_best"][::-1])
print("Pooled OOF confusion matrix of the full model (compare with Table 14):")
print(cm_full)

table15b_best.to_csv("table15b_best.csv")
table15b_last.to_csv("table15b_last.csv")
table15a_jaffe.to_csv("table15a_jaffe.csv")


# ------------------------- Table 17 -------------------------
if RUN_TABLE17:
    print("\n" + "=" * 70)
    print(f"  TABLE 17 — seeds {T17_SEEDS}: DenseNet121 (a) vs DRAG-Net (e)")
    print("=" * 70)
    seed_runs = {"a": {}, "e": {}}
    for seed in T17_SEEDS:
        for stage in ("a", "e"):
            if seed == TABLE_SEED:                     # reuse the Table 15b runs
                seed_runs[stage][seed] = results_15b[stage]
            else:
                print(f"\n▶ Stage {stage}, seed {seed}")
                seed_runs[stage][seed] = run_stage(stage, seed)

    def build_table17(which="best"):
        rows, per_seed = [], {"a": [], "e": []}
        for seed in T17_SEEDS:
            cells = {}
            for stage in ("a", "e"):
                accs = [m["Accuracy (%)"] for m in seed_runs[stage][seed][which]]
                m, s = mean_sd(accs)
                per_seed[stage].append(m)
                cells[stage] = f"{m:.2f} ± {s:.2f}"
            rows.append({"Seed": f"Seed {seed}",
                         "DenseNet121 Baseline": cells["a"],
                         "DRAG-Net (Proposed)": cells["e"]})
        ma, sa = mean_sd(per_seed["a"])
        me, se = mean_sd(per_seed["e"])
        t, p = stats.ttest_rel(per_seed["e"], per_seed["a"])
        rows.append({"Seed": "Mean ± SD (across seeds)",
                     "DenseNet121 Baseline": f"{ma:.2f} ± {sa:.2f}",
                     "DRAG-Net (Proposed)": f"{me:.2f} ± {se:.2f}"})
        rows.append({"Seed": f"Paired t-test (df = {len(T17_SEEDS)-1})",
                     "DenseNet121 Baseline": "—",
                     "DRAG-Net (Proposed)": f"t = {t:.2f}, p = {p:.3f}"})
        return pd.DataFrame(rows).set_index("Seed")

    table17_best = build_table17("best")
    table17_last = build_table17("last")
    print("\nTable 17 — per-seed mean ± SD over folds (BEST test epoch)")
    display(table17_best)
    print("\nTable 17 — per-seed mean ± SD over folds (LAST epoch)")
    display(table17_last)
    table17_best.to_csv("table17_best.csv")
    table17_last.to_csv("table17_last.csv")
